# Install & imports

In [ ]:
!pip install -q torch-geometric

import torch
import torch.nn as nn
import torch.nn.functional as torch_F
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00
Device: cpu


# Reconnect paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/PD_EEG_Thesis_Features_v2')
MODEL_DIR    = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Model definitions will be saved for reuse in Notebooks 6-9.')

Mounted at /content/drive
Model definitions will be saved for reuse in Notebooks 6-9.


# Architecture hyperparameters

In [ ]:
N_TIMESTEPS = 6          # non-overlapping 1s sub-windows
N_FEATURES  = 17          # finalized feature count

HIDDEN_DIM  = 48          # sized for smaller feature input, not raw-voltage's 300-dim
N_HEADS     = 4
DROPOUT     = 0.25

print(f'N_FEATURES={N_FEATURES}  HIDDEN_DIM={HIDDEN_DIM}  N_HEADS={N_HEADS}  DROPOUT={DROPOUT}')

N_FEATURES=17  HIDDEN_DIM=48  N_HEADS=4  DROPOUT=0.25


# Model definitions — TGCN, TGIN, TGAT with attention-weighted pooling

In [ ]:
class AttentionPool(nn.Module):
    """Learnable per-channel attention weighting — lets the model learn
    which channels matter most, rather than averaging all 60 uniformly."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, emb):
        scores = self.attn(emb)
        weights = torch_F.softmax(scores, dim=1)
        pooled = (weights * emb).sum(dim=1)
        return pooled, weights.squeeze(-1)


class TGCN(nn.Module):
    def __init__(self, n_features, hidden_dim, dropout):
        super().__init__()
        self.gc_weight  = nn.Linear(n_features, hidden_dim, bias=False)
        self.pool       = AttentionPool(hidden_dim)
        self.gru        = nn.GRU(hidden_dim, hidden_dim, batch_first=True, num_layers=2, dropout=dropout)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 2)

    def graph_conv(self, x, adj):
        deg = adj.sum(dim=-1, keepdim=True).clamp(min=1)
        x   = torch.bmm(adj / deg, x)
        return torch_F.relu(self.gc_weight(x))

    def forward(self, node_feat, adj):
        B, T, N, Fdim = node_feat.shape
        embs = []
        for t in range(T):
            h = self.graph_conv(node_feat[:, t], adj)
            pooled, _ = self.pool(h)
            embs.append(pooled)
        seq, _ = self.gru(torch.stack(embs, dim=1))
        return self.classifier(self.dropout(seq[:, -1, :]))


class TGIN(nn.Module):
    def __init__(self, n_features, hidden_dim, dropout):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(n_features, hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, hidden_dim))
        self.eps  = nn.Parameter(torch.zeros(1))
        self.pool = AttentionPool(hidden_dim)
        self.gru  = nn.GRU(hidden_dim, hidden_dim, batch_first=True, num_layers=2, dropout=dropout)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 2)

    def graph_gin(self, x, adj):
        agg = torch.bmm(adj, x)
        return self.mlp((1 + self.eps) * x + agg)

    def forward(self, node_feat, adj):
        B, T, N, Fdim = node_feat.shape
        embs = []
        for t in range(T):
            h = self.graph_gin(node_feat[:, t], adj)
            pooled, _ = self.pool(h)
            embs.append(pooled)
        seq, _ = self.gru(torch.stack(embs, dim=1))
        return self.classifier(self.dropout(seq[:, -1, :]))


class TGAT(nn.Module):
    """Plain mean-pool, not AttentionPool — TGAT already has its own internal
    multi-head attention; a second pooling-attention layer proved redundant
    in earlier testing on this exact task."""
    def __init__(self, n_features, hidden_dim, n_heads, dropout):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = hidden_dim // n_heads
        self.W_q = nn.Linear(n_features, hidden_dim, bias=False)
        self.W_k = nn.Linear(n_features, hidden_dim, bias=False)
        self.W_v = nn.Linear(n_features, hidden_dim, bias=False)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)
        self.gru        = nn.GRU(hidden_dim, hidden_dim, batch_first=True, num_layers=2, dropout=dropout)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 2)

    def graph_attention(self, x, adj):
        B, N, Fdim = x.shape
        H, D = self.n_heads, self.head_dim
        Q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        K = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        V = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (D ** 0.5)
        mask   = (adj == 0).unsqueeze(1).expand_as(scores)
        scores = scores.masked_fill(mask, float('-inf'))
        scores = torch.nan_to_num(scores, nan=0.0, neginf=-1e9)
        attn   = torch_F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, -1)
        return torch_F.relu(self.W_o(out))

    def forward(self, node_feat, adj):
        B, T, N, Fdim = node_feat.shape
        embs = [self.graph_attention(node_feat[:, t], adj).mean(dim=1) for t in range(T)]
        seq, _ = self.gru(torch.stack(embs, dim=1))
        return self.classifier(self.dropout(seq[:, -1, :]))


def build_model(name, n_features=N_FEATURES):
    if name == 'TGCN': return TGCN(n_features, HIDDEN_DIM, DROPOUT).to(DEVICE)
    if name == 'TGIN': return TGIN(n_features, HIDDEN_DIM, DROPOUT).to(DEVICE)
    if name == 'TGAT': return TGAT(n_features, HIDDEN_DIM, N_HEADS, DROPOUT).to(DEVICE)

print('Models defined: TGCN, TGIN (attention-pooled), TGAT (mean-pooled).')

Models defined: TGCN, TGIN (attention-pooled), TGAT (mean-pooled).


# Sanity check — verify each model runs on dummy data

In [ ]:
dummy_nf  = torch.randn(4, N_TIMESTEPS, 60, N_FEATURES).to(DEVICE)
dummy_adj = torch.rand(4, 60, 60).to(DEVICE)
dummy_adj = (dummy_adj + dummy_adj.transpose(1,2)) / 2   # symmetric, like real adjacency

for name in ['TGCN', 'TGIN', 'TGAT']:
    model = build_model(name)
    out = model(dummy_nf, dummy_adj)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{name}: output shape {out.shape}  |  {n_params:,} parameters')
    assert out.shape == (4, 2), f'{name} output shape mismatch!'

print('\nAll three models pass the sanity check.')

TGCN: output shape torch.Size([4, 2])  |  29,187 parameters
TGIN: output shape torch.Size([4, 2])  |  31,588 parameters
TGAT: output shape torch.Size([4, 2])  |  33,122 parameters

All three models pass the sanity check.
